# Feature Engineering

In [44]:
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect('../diabetes-readmission.db')

df = pd.read_sql("""
    SELECT * FROM encounters
""", conn)

df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Reasoning for later

Start by dropping the columns I know I won't be needing

In [45]:
columns_to_drop = ['weight', 'medical_specialty', 'payer_code', 'max_glu_serum', 'encounter_id']

df.drop(columns=columns_to_drop, inplace=True)
df.shape

(101766, 45)

## Reasoning For Later

Deduplicate the encounters to unique patients only. This will reduce the memory held as well as reduce time for computations later.

In [46]:
df.drop_duplicates(subset='patient_nbr', keep='first', inplace=True)
df.drop(columns='patient_nbr', inplace=True)
df.reset_index()
df.shape

(71518, 44)

### Reasoning For Later

Modify the columns I kept that have missing values

In [47]:
def replace_missing_data(entry):
    entry = str(entry).strip()
    if entry == "?":
        return "Unknown"
    return entry

def categorize_icd9(code):
    code = str(code).strip()
    
    if code == '?':
        return "Unknown"
    
    if code.startswith('V'):
        return "Supplementary"
    if code.startswith("E"):
        return "External"
    
    try:
        num = float(code)
    except ValueError:
        return "Other"
    
    if num < 140:
        return "Infectious Disease"
    elif num >=140 and num < 240:
        return "Neoplasms"
    elif num >=240 and num < 280:
        if num >= 250 and num < 251:
            return "Diabetes"
        return "Endocrine"
    elif num >=280 and num < 290:
        return "Blood Disease"
    elif num >=290 and num < 320:
        return "Behavioral"
    elif num >=320 and num < 390:
        return "Nervous System"
    elif num >=390 and num < 460:
        return "Circulatory"
    elif num >=460 and num < 520:
        return "Respiratory"
    elif num >=520 and num < 580:
        return "Digestive"
    elif num >=580 and num < 630:
        return "Genitourinary"
    elif num >=630 and num < 680:
        return "Pregnancy"
    elif num >=680 and num < 710:
        return "Skin"
    elif num >=710 and num < 740:
        return "Musculoskeletal"
    elif num >=740 and num < 760:
        return "Congenital Anomalies"
    elif num >=760 and num < 780:
        return "Perinatal"
    elif num >=780 and num < 800:
        return "Ill-defined Conditions"
    elif num >=800 and num < 1000:
        return "Injury and Poisoning"
    
    

    return "Other"


In [48]:
df['race_cat'] = df['race'].apply(replace_missing_data)
df['diag_1_cat'] = df['diag_1'].apply(categorize_icd9)
df['diag_2_cat'] = df['diag_2'].apply(categorize_icd9)
df['diag_3_cat'] = df['diag_3'].apply(categorize_icd9)

print(df['diag_1_cat'].value_counts())
print(df['race_cat'].value_counts(normalize=True))

diag_1_cat
Circulatory               21821
Respiratory                6736
Digestive                  6403
Diabetes                   5805
Ill-defined Conditions     5530
Injury and Poisoning       4779
Musculoskeletal            4080
Genitourinary              3488
Neoplasms                  2742
Endocrine                  1880
Infectious Disease         1818
Skin                       1789
Behavioral                 1548
Supplementary               927
Nervous System              876
Blood Disease               657
Pregnancy                   586
Congenital Anomalies         41
Unknown                      11
External                      1
Name: count, dtype: int64
race_cat
Caucasian          0.747938
AfricanAmerican    0.180192
Unknown            0.027238
Hispanic           0.021211
Other              0.016471
Asian              0.006949
Name: proportion, dtype: float64
